# 01 · Qué se midió y qué puede entrar en los modelos
Se reconstruye la matriz en mM a partir de las filas auditadas y se verifica cada posición contra el Excel correspondiente. Una fila es un cultivo o una condición; una columna, una señal analítica identificada. Ejecute las celdas desde la raíz del repositorio. Para reconstruir **la auditoría de origen** desde SQLite, consulte la versión del 16 de noviembre en [histórico](historico/01_auditoria_preparacion.ipynb).

In [1]:
import pandas as pd
from IPython.display import display
from ampliacion.preparar import SALIDA, preparar

ishii, mmc2, wt_adicionales, metadatos, referencias = preparar()
display(pd.DataFrame({
    "cultivos o condiciones": [len(ishii), len(mmc2), len(wt_adicionales)],
    "rol": ["prueba y entrenamiento", "solo entrenamiento B", "solo sensibilidad; grupo WT"],
}, index=["Ishii a D=0,2", "Gerosa MMC2", "Ishii WT a otras diluciones"]))

,cultivos o condiciones,rol
"Ishii a D=0,2",30,prueba y entrenamiento
Gerosa MMC2,8,solo entrenamiento B
Ishii WT a otras diluciones,4,solo sensibilidad; grupo WT


## Qué se puede comparar
Las veinte correspondencias nominales se mantienen como posibles objetivos. Las siete señales nuevas conservan su etiqueta Ishii: una contraparte propuesta no equivale a una medición utilizable. La columna de disponibilidad distingue ambas cosas. El [diccionario completo](ampliacion/resultados/diccionario.csv) incluye compartimento, método, unidades, referencias y filas de los originales.

In [2]:
diccionario = pd.read_csv(SALIDA / "diccionario.csv")
display(diccionario[["variable", "identidad_Ishii", "protocolo_Ishii", "n_Ishii_30",
                     "n_WT_extra", "etiqueta_MMC2", "n_MMC2_utilizables", "rol"]])

,variable,identidad_Ishii,protocolo_Ishii,n_Ishii_30,n_WT_extra,etiqueta_MMC2,n_MMC2_utilizables,rol
0,pyr,Pyruvate,Anion,14,1,pyr,7,objetivo_y_predictor
1,fum,Fumarate,Anion,23,4,fum,8,objetivo_y_predictor
2,succ,Succinate,Anion,27,4,succ,7,objetivo_y_predictor
3,akg,2-Oxoglutarate,Anion,11,1,akg,8,objetivo_y_predictor
4,pep,Phosphoenolpyruvate,Anion,22,4,pep,8,objetivo_y_predictor
5,dhap,Dihydroxyacetone phosphate,Anion,8,4,dhap,8,objetivo_y_predictor
6,s7p,Sedoheptulose-7-phosphate,Anion,29,4,s7p,8,objetivo_y_predictor
7,ade,Adenine,Cation,30,4,ade,8,objetivo_y_predictor
8,cit,Citrate,Nucleotide,12,0,cit,8,objetivo_y_predictor
9,cAMP,cAMP,Nucleotide,29,4,cAMP,8,objetivo_y_predictor


## Seguir un valor hasta su origen
La concentración de Ishii se lee en mM. La de MMC2 y su SD se dividen por el mismo volumen citosólico de 2,3 mL/g de peso seco. Una ausencia permanece vacía; un cero medido se conserva. Estas tres filas ilustran la diferencia entre una señal de Ishii y una medición compartida.

In [3]:
traza = pd.read_csv(SALIDA / "trazabilidad.csv")
ejemplo = ((traza.sample_key == "KO01") & traza.variable.isin(["ish_g6p", "fum"])) | (
    (traza.sample_key == "MMC2_GLUCOSE") & (traza.variable == "fum"))
display(traza.loc[ejemplo, ["sample_key", "variable", "source_id", "source_sheet", "source_cell",
                            "unit_original", "divisor_conversion", "valor_mM", "SD_mM"]])

,sample_key,variable,source_id,source_sheet,source_cell,unit_original,divisor_conversion,valor_mM,SD_mM
139,MMC2_GLUCOSE,fum,MMC2,Metabolite concentrations,G38,umol gCDW^-1,2.3,0.548242,0.087006
194,KO01,fum,ISHII,metabolites,B29,mM,1.0,0.056300,NaN
568,KO01,ish_g6p,ISHII,metabolites,B174,mM,1.0,0.138000,NaN


## ¿Qué fisiología se puede enlazar con cada cultivo?
La dilución, la captación y el flujo son cantidades distintas de una concentración. Se leen los ID de **cada hoja**: por ejemplo, RF04 tiene un flujo relativo, pero no una captación publicada en la hoja de tasas. La siguiente tabla cuenta valores disponibles, incluidos los ceros publicados. En MMC2 el flujo específico de esta tabla queda vacío porque no se equipara automáticamente con un módulo de Gerosa.

In [4]:
from ampliacion.fisiologia import auditar

tasas = auditar(ishii.index, wt_adicionales.index, mmc2.index, metadatos)
display(tasas.groupby("cohorte")[["D", "mu", "q_sustrato", "v_rel", "v_abs"]].count().rename(
    columns={"D": "Dilución", "mu": "Crecimiento (Gerosa)", "q_sustrato": "Captación",
             "v_rel": "G6P → 6PG relativo", "v_abs": "G6P → 6PG absoluto"}))

,Dilución,Crecimiento (Gerosa),Captación,G6P → 6PG relativo,G6P → 6PG absoluto
cohorte,,,,,
Ishii30,30,0,27,27,24
MMC2,0,8,8,0,0
WT4,4,0,4,4,4


## Bloque 1 del plan: inventario y panel de referencia

El panel de referencia tiene 20 objetivos documentados y la matriz contiene además siete predictores exclusivos de Ishii. La siguiente tabla cuenta filas analíticas, no moléculas independientes. Las candidatas pendientes de revisión no se incorporan automáticamente. [Ver decisiones y trazabilidad](ampliacion/BLOQUES_1_4.md).

In [ ]:
import pandas as pd
from ampliacion.preparar import SALIDA

inventario = pd.read_csv(SALIDA / "bloques_1_4" / "inventario_resumen.csv")
inventario.groupby("decision").filas.sum().sort_values(ascending=False)

Continúe con [02 · Evaluación e imputación](02_imputacion_evaluacion.ipynb). La elegibilidad se recalcula sin las muestras de prueba en cada bloque; el recuento de esta página es descriptivo.